In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2009_2010"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "MCQ_F.xpt"
csv_file = DATA_DIR / "MCQ_F_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched pandas-decoded XPT values.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2009-2010 MCQ_F XPT -> CSV")
print("=" * 75)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "MCQ010",
    "MCQ025",
    "MCQ035",
    "MCQ040",
    "MCQ050",
    "MCQ051",
    "MCQ053",
    "MCQ070",
    "MCQ080",
    "MCQ082",
    "MCQ086",
    "MCQ092",
    "MCD093",
    "MCQ140",
    "MCQ149",
    "MCQ160A",
    "MCQ180A",
    "MCQ191",
    "MCQ160N",
    "MCQ180N",
    "MCQ160B",
    "MCQ180B",
    "MCQ160C",
    "MCQ180C",
    "MCQ160D",
    "MCQ180D",
    "MCQ160E",
    "MCQ180E",
    "MCQ160F",
    "MCQ180F",
    "MCQ160G",
    "MCQ180G",
    "MCQ160M",
    "MCQ170M",
    "MCQ180M",
    "MCQ160K",
    "MCQ170K",
    "MCQ180K",
    "MCQ160L",
    "MCQ170L",
    "MCQ180L",
    "MCQ220",
    "MCQ230A",
    "MCQ230B",
    "MCQ230C",
    "MCQ230D",
    "MCQ240A",
    "MCQ240AA",
    "MCQ240B",
    "MCQ240BB",
    "MCQ240C",
    "MCQ240CC",
    "MCQ240D",
    "MCQ240DD",
    "MCQ240DK",
    "MCQ240E",
    "MCQ240F",
    "MCQ240G",
    "MCQ240H",
    "MCQ240I",
    "MCQ240J",
    "MCQ240K",
    "MCQ240L",
    "MCQ240M",
    "MCQ240N",
    "MCQ240O",
    "MCQ240P",
    "MCQ240Q",
    "MCQ240R",
    "MCQ240S",
    "MCQ240T",
    "MCQ240U",
    "MCQ240V",
    "MCQ240W",
    "MCQ240X",
    "MCQ240Y",
    "MCQ240Z",
    "MCQ300A",
    "MCQ300B",
    "MCQ300C"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "MCQ_F column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 10109:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 81:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 10,109 rows x 81 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# IMPORTANT:
#
# Never globally replace this value.
#
# First validate:
#
#     variable
#     count
#     CDC valid range / code meaning
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


# ============================================================
# 5. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Validated against:
#
#   attached MCQ_F.xpt
#   CDC 2009-2010 MCQ_F dictionary
#
#
# MCQ180M:
# Age when told had thyroid problem
#
# CDC valid range:
#     0-79
#
#
# MCQ180K:
# Age when told had chronic bronchitis
#
# CDC valid range:
#     0-78
#
#
# MCQ240DD:
# Age other type cancer first diagnosed
#
# CDC valid range:
#     0-77
#
# Therefore zero is legitimate in all three.
#

EXPECTED_TINY_COUNTS = {
    "MCQ180M": 1,
    "MCQ180K": 1,
    "MCQ240DD": 1
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated MCQ_F pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 3:

    raise ValueError(
        f"Unexpected total tiny-value count: {tiny_total}"
    )


print(
    "\nPASS: Exact 3-value tiny-zero pattern validated."
)


# ============================================================
# 6. RESTORE ONLY VALIDATED ZERO VALUES
# ============================================================

print("\n--- ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    tiny_mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        tiny_mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: unexpected tiny-value count."
        )


    df.loc[
        tiny_mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 3:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    correction_total,
    "validated zeros restored."
)


# ============================================================
# 7. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# dataset should contain.
#
# ONLY these positions change:
#
#     TINY_VALUE -> 0
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col].eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 8. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# CRITICAL FIDELITY RULE:
#
# Never round genuine decimal values.
#
# Perform this check BEFORE Int64 conversion.
#
# MCQ_F contains no genuine fractional variables.
#
# If any decimal variable unexpectedly appears,
# STOP rather than round it.
#
# This same safeguard protects Dietary, Lab,
# Examination and sample-weight decimal fields.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"{col}: "
            f"{int(fractional_mask.sum()):,} "
            "genuine fractional values"
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal variables detected. "
        "They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional values "
    "exist in MCQ_F."
)


# ============================================================
# 9. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
    or
    df["SEQN"].min()
    != 51624
    or
    df["SEQN"].max()
    != 62160
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 10. FORMAT QUESTIONNAIRE VALUES WITHOUT .0
# ============================================================

#
# After:
#
#   validated tiny-zero correction
#   genuine-decimal check
#
# all 81 MCQ_F variables are proven whole-number
# questionnaire variables.
#
# NO .round() is used.
#

for col in df.columns:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_number_check = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not np.all(
            whole_number_check
        ):

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 81 MCQ_F fields formatted "
    "without unnecessary .0."
)


# ============================================================
# 11. CDC CHECK - MCQ010
# ============================================================

#
# Ever been told you have asthma
#
# CDC:
#
# 1 = Yes          1,451
# 2 = No           8,641
# 7 = Refused          1
# 9 = Don't know      16
# Missing              0
#

EXPECTED_MCQ010 = {
    1: 1451,
    2: 8641,
    7: 1,
    9: 16
}


actual = (
    df["MCQ010"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_MCQ010
    or
    int(
        df["MCQ010"]
        .isna()
        .sum()
    )
    != 0
):

    raise ValueError(
        "MCQ010 differs from CDC."
    )


print(
    "PASS: MCQ010 matches CDC."
)


# ============================================================
# 12. CDC CHECK - MCQ025
# ============================================================

#
# Age when first had asthma
#
# CDC:
#
# 1-78   = 1,433
# 80     = 6
# 99999  = 12
# Missing = 8,658
#

s = df["MCQ025"]


if not (
    int(
        s.between(
            1,
            78
        ).sum()
    )
    == 1433
    and
    int(
        s.eq(80)
        .sum()
    )
    == 6
    and
    int(
        s.eq(77777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(99999)
        .sum()
    )
    == 12
    and
    int(
        s.isna()
        .sum()
    )
    == 8658
):

    raise ValueError(
        "MCQ025 differs from CDC."
    )


print(
    "PASS: MCQ025 matches CDC."
)


# ============================================================
# 13. CDC CHECK - MCQ160A
# ============================================================

#
# Doctor ever said participant had arthritis
#
# CDC:
#
# 1 = Yes          1,674
# 2 = No           4,527
# 9 = Don't know      17
# Missing          3,891
#

EXPECTED_MCQ160A = {
    1: 1674,
    2: 4527,
    9: 17
}


actual = (
    df["MCQ160A"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_MCQ160A
    or
    int(
        df["MCQ160A"]
        .isna()
        .sum()
    )
    != 3891
):

    raise ValueError(
        "MCQ160A differs from CDC."
    )


print(
    "PASS: MCQ160A matches CDC."
)


# ============================================================
# 14. CDC CHECK - MCQ180M
# ============================================================

#
# Age when told participant had thyroid problem
#
# CDC:
#
# 0-79   = 583
# 80     = 17
# 99999  = 8
# Missing = 9,501
#
# One raw tiny XPORT value represents age 0.
#

s = df["MCQ180M"]


print("\n--- MCQ180M ---")


print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


if not (
    int(
        s.between(
            0,
            79
        ).sum()
    )
    == 583
    and
    int(
        s.eq(0)
        .sum()
    )
    == 1
    and
    int(
        s.eq(80)
        .sum()
    )
    == 17
    and
    int(
        s.eq(77777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(99999)
        .sum()
    )
    == 8
    and
    int(
        s.isna()
        .sum()
    )
    == 9501
):

    raise ValueError(
        "MCQ180M differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: MCQ180M matches CDC."
)


# ============================================================
# 15. CDC CHECK - MCQ180K
# ============================================================

#
# Age when told participant had chronic bronchitis
#
# CDC:
#
# 0-78   = 302
# 80     = 4
# 99999  = 5
# Missing = 9,798
#
# One raw tiny value represents age 0.
#

s = df["MCQ180K"]


print("\n--- MCQ180K ---")


print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


if not (
    int(
        s.between(
            0,
            78
        ).sum()
    )
    == 302
    and
    int(
        s.eq(0)
        .sum()
    )
    == 1
    and
    int(
        s.eq(80)
        .sum()
    )
    == 4
    and
    int(
        s.eq(77777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(99999)
        .sum()
    )
    == 5
    and
    int(
        s.isna()
        .sum()
    )
    == 9798
):

    raise ValueError(
        "MCQ180K differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: MCQ180K matches CDC."
)


# ============================================================
# 16. CDC CHECK - MCQ220
# ============================================================

#
# Ever told participant had cancer or malignancy
#
# CDC:
#
# 1 = Yes            620
# 2 = No           5,591
# 9 = Don't know       7
# Missing          3,891
#

EXPECTED_MCQ220 = {
    1: 620,
    2: 5591,
    9: 7
}


actual = (
    df["MCQ220"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_MCQ220
    or
    int(
        df["MCQ220"]
        .isna()
        .sum()
    )
    != 3891
):

    raise ValueError(
        "MCQ220 differs from CDC."
    )


print(
    "PASS: MCQ220 matches CDC."
)


# ============================================================
# 17. CDC CHECK - MCQ240DD
# ============================================================

#
# Age when "other" type cancer first diagnosed
#
# CDC:
#
# 0-77   = 31
# 80     = 2
# Missing = 10,076
#
# One raw tiny value represents age 0.
#

s = df["MCQ240DD"]


print("\n--- MCQ240DD ---")


print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


if not (
    int(
        s.between(
            0,
            77
        ).sum()
    )
    == 31
    and
    int(
        s.eq(0)
        .sum()
    )
    == 1
    and
    int(
        s.eq(80)
        .sum()
    )
    == 2
    and
    int(
        s.eq(77777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(99999)
        .sum()
    )
    == 0
    and
    int(
        s.isna()
        .sum()
    )
    == 10076
):

    raise ValueError(
        "MCQ240DD differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: MCQ240DD matches CDC."
)


# ============================================================
# 18. CDC FAMILY HISTORY CHECKS
# ============================================================

CDC_FAMILY_HISTORY = {

    "MCQ300A": {
        "values": {
            1: 712,
            2: 5333,
            9: 173
        },
        "missing": 3891
    },

    "MCQ300B": {
        "values": {
            1: 1810,
            2: 6811,
            7: 3,
            9: 189
        },
        "missing": 1296
    },

    "MCQ300C": {
        "values": {
            1: 2348,
            2: 3723,
            7: 2,
            9: 145
        },
        "missing": 3891
    }
}


print("\n--- CDC FAMILY HISTORY CHECKS ---")


for col, expected in CDC_FAMILY_HISTORY.items():

    actual_values = (
        df[col]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if actual_values != expected["values"]:

        raise ValueError(
            f"{col} frequencies differ from CDC.\n"
            f"Expected: {expected['values']}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected["missing"]:

        raise ValueError(
            f"{col} missing count differs from CDC.\n"
            f"Expected: {expected['missing']}\n"
            f"Actual:   {actual_missing}"
        )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 19. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Expected differences from raw pandas XPT:
#
# MCQ180M:
#     1 tiny value -> 0
#
# MCQ180K:
#     1 tiny value -> 0
#
# MCQ240DD:
#     1 tiny value -> 0
#
# NOTHING ELSE may change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(failures)
    )


print(
    "PASS: All 81 variables match expected "
    "corrected XPT values exactly."
)


# ============================================================
# 20. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format.
# NO rounding.
#
# MCQ_F has been proven to contain only
# whole-number fields after zero restoration.
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 21. RE-READ EXPORTED CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 22. EXACT CORRECTED XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 81 CSV columns match "
    "expected corrected XPT values exactly."
)


# ============================================================
# 23. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 24. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in csv_df.select_dtypes(
    include=[np.number]
).columns:

    tiny_csv_count += int(
        csv_df[col]
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 25. RAW CSV .0 FORMAT CHECK
# ============================================================

#
# MCQ_F contains NO genuine decimal variables.
#
# Therefore ".0" should not occur anywhere in this
# particular final CSV.
#
# IMPORTANT:
#
# Do NOT apply this same global rule to Dietary,
# continuous Lab or Examination datasets.
#

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    raise ValueError(
        "Unexpected .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 formatting."
)


# ============================================================
# 26. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 27. FINAL CSV SEQN CHECK
# ============================================================

print("\n--- FINAL CSV SEQN CHECK ---")


if (
    csv_df["SEQN"].min()
    != 51624
    or
    csv_df["SEQN"].max()
    != 62160
    or
    int(
        csv_df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        csv_df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN is valid."
)


# ============================================================
# 28. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)


print(
    "MCQ_F CONVERSION COMPLETE"
)


print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "MCQ180M zero restorations:",
    1
)


print(
    "MCQ180K zero restorations:",
    1
)


print(
    "MCQ240DD zero restorations:",
    1
)


print(
    "Total validated zero restorations:",
    correction_total
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "XPT -> CSV unexpected differences: 0 expected"
)


print(
    "Missing-value differences: 0 expected"
)


print(
    "Tiny values remaining: 0 expected"
)


print(
    "Unnecessary .0: 0 expected"
)


print(
    "Scientific notation: 0 expected"
)

NHANES 2009-2010 MCQ_F XPT -> CSV
Rows: 10,109
Columns: 81
PASS: Structure = 10,109 rows x 81 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 3

Tiny values by variable:
MCQ180M     1
MCQ180K     1
MCQ240DD    1

PASS: Exact 3-value tiny-zero pattern validated.

--- ZERO RESTORATION ---
MCQ180M: 1 restored to 0
MCQ180K: 1 restored to 0
MCQ240DD: 1 restored to 0

PASS: 3 validated zeros restored.

--- FRACTIONAL VALUE CHECK ---
PASS: No genuine fractional values exist in MCQ_F.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 51624
Maximum: 62160
PASS: SEQN validated.

PASS: All 81 MCQ_F fields formatted without unnecessary .0.
PASS: MCQ010 matches CDC.
PASS: MCQ025 matches CDC.
PASS: MCQ160A matches CDC.

--- MCQ180M ---
Zero: 1
PASS: MCQ180M matches CDC.

--- MCQ180K ---
Zero: 1
PASS: MCQ180K matches CDC.
PASS: MCQ220 matches CDC.

--- MCQ240DD ---
Zero: 1
PASS: MCQ240DD matches CDC.

--- CDC FAMILY HISTORY CHECKS ---
PASS: MCQ300A matches CDC